# Madgwick IMU Pipeline Validation Notebook

This notebook validates the **XIAO Seeed IMU → Madgwick orientation** pipeline.

Use `.py` scripts for real-time serial reading, and use this notebook for offline validation, plotting, and documentation.

Checks included:

- timestamp and sampling interval
- accelerometer unit and gravity magnitude
- gyroscope unit, noise, and approximate angle integration
- coordinate-system worksheet
- Madgwick quaternion / Euler-angle output
- quaternion norm
- gravity estimate vs measured acceleration
- user acceleration
- static-test and manoeuvre-level summary

Project scope at this stage:

> Use the supervisor-provided Madgwick filter as a baseline/reference estimator. Confirm units, coordinate-system convention, and BPPV manoeuvre-level orientation interpretation before implementing Kalman/EKF comparison.

IMU-only limitation: without magnetometer, **yaw/heading can drift**. Roll and pitch can be corrected using gravity.

## 0. Expected folder structure

```text
imu_project/
├── madgwick_filter.py
├── data/
│   ├── static_30s_raw.csv
│   ├── static_30s_madgwick.csv       # optional; notebook can generate this
│   ├── roll_test_raw.csv
│   ├── pitch_test_raw.csv
│   └── epley_like_raw.csv
└── validate_madgwick_pipeline.ipynb
```

Expected raw CSV format:

```text
timestamp_ms, ax, ay, az, gx, gy, gz
```

or:

```text
timestamp_s, ax, ay, az, gx, gy, gz
```

## 1. Setup

Edit `DATASET_NAME`, units, and paths below.

Standard internal units:

| Quantity | Internal unit |
|---|---|
| time | seconds |
| acceleration | m/s² |
| gyroscope | rad/s |
| Euler angles | radians internally, degrees for plots |
| quaternion | dimensionless unit quaternion |

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# -------------------------
# USER CONFIGURATION
# -------------------------
DATASET_NAME = "static_30s"   # e.g. static_30s, roll_test, pitch_test, yaw_test, epley_like
DATA_DIR = Path("data")

RAW_PATH = DATA_DIR / f"{DATASET_NAME}_raw.csv"
OUT_PATH = DATA_DIR / f"{DATASET_NAME}_madgwick.csv"

# Set these based on your Arduino/IMU library output.
# TIME_UNIT: "ms" or "s"
# ACCEL_UNIT: "m/s2" or "g"
# GYRO_UNIT: "rad/s" or "deg/s"
TIME_UNIT = "ms"
ACCEL_UNIT = "m/s2"
GYRO_UNIT = "deg/s"     # many Arduino libraries output deg/s; change if already rad/s

# Used only if the notebook replays raw data through madgwick_filter.py
BETA = 0.1
GRAVITY = 9.80665

print("Raw file:", RAW_PATH)
print("Madgwick output file:", OUT_PATH)

## 2. Load raw IMU data and standardize units

The output dataframe `raw` will always use:

```text
timestamp_s, ax, ay, az, gx, gy, gz
```

where acceleration is in `m/s²` and gyroscope is in `rad/s`.

In [ ]:
def load_raw_imu(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Update RAW_PATH or put the CSV in the data folder."
        )

    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    # Edit this if your CSV uses different column names.
    rename_map = {
        "time": "timestamp_s",
        "timestamp": "timestamp_s",
        "t": "timestamp_s",
        "timestamp_ms": "timestamp_ms",
        "accel_x": "ax",
        "accel_y": "ay",
        "accel_z": "az",
        "gyro_x": "gx",
        "gyro_y": "gy",
        "gyro_z": "gz",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # Time conversion
    if "timestamp_s" not in df.columns:
        if "timestamp_ms" not in df.columns:
            raise ValueError(f"Need timestamp_s or timestamp_ms. Current columns: {list(df.columns)}")
        if TIME_UNIT == "ms":
            df["timestamp_s"] = df["timestamp_ms"] / 1000.0
        elif TIME_UNIT == "s":
            df["timestamp_s"] = df["timestamp_ms"]
        else:
            raise ValueError("TIME_UNIT must be 'ms' or 's'.")

    required = ["timestamp_s", "ax", "ay", "az", "gx", "gy", "gz"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}. Current columns: {list(df.columns)}")

    df = df[required].copy().dropna().reset_index(drop=True)

    # Acceleration conversion to m/s²
    if ACCEL_UNIT == "g":
        df[["ax", "ay", "az"]] *= GRAVITY
    elif ACCEL_UNIT != "m/s2":
        raise ValueError("ACCEL_UNIT must be 'm/s2' or 'g'.")

    # Gyro conversion to rad/s
    if GYRO_UNIT == "deg/s":
        df[["gx", "gy", "gz"]] *= math.pi / 180.0
    elif GYRO_UNIT != "rad/s":
        raise ValueError("GYRO_UNIT must be 'rad/s' or 'deg/s'.")

    # Start time at zero for plotting.
    df["timestamp_s"] = df["timestamp_s"] - df["timestamp_s"].iloc[0]
    return df

raw = load_raw_imu(RAW_PATH)
raw["timestamp_s"] = raw["timestamp_s"] - raw["timestamp_s"].iloc[0]
raw.head()

## 3. Timestamp and sampling-rate check

Check that:

- timestamps are increasing
- `dt` is positive
- sampling rate is stable enough
- no big `dt` spikes appear

In [ ]:
t = raw["timestamp_s"].to_numpy()
dt = np.diff(t)

print(f"Number of samples: {len(raw)}")
print(f"Duration: {t[-1] - t[0]:.3f} s")
print(f"Mean dt: {np.mean(dt):.6f} s")
print(f"Median dt: {np.median(dt):.6f} s")
print(f"Min dt: {np.min(dt):.6f} s")
print(f"Max dt: {np.max(dt):.6f} s")
print(f"Mean sampling rate: {1 / np.mean(dt):.2f} Hz")
print(f"Median sampling rate: {1 / np.median(dt):.2f} Hz")
print("Any non-positive dt?", np.any(dt <= 0))

plt.figure(figsize=(10, 4))
plt.plot(t[1:], dt)
plt.xlabel("Time [s]")
plt.ylabel("dt [s]")
plt.title(f"{DATASET_NAME}: Sampling interval")
plt.grid(True)
plt.show()

## 4. Accelerometer unit and gravity check

For a stationary IMU:

```text
sqrt(ax^2 + ay^2 + az^2) ≈ 9.81 m/s²
```

Interpretation:

| Static acceleration norm | Likely meaning |
|---:|---|
| around 9.8 | acceleration is already m/s² |
| around 1.0 | acceleration is in g; set `ACCEL_UNIT = "g"` |
| very different or unstable | check scaling, calibration, or whether the sensor was moving |

In [ ]:
raw = raw.copy()
raw["accel_norm"] = np.sqrt(raw["ax"]**2 + raw["ay"]**2 + raw["az"]**2)

print("Acceleration norm statistics [m/s²]:")
print(raw["accel_norm"].describe())

plt.figure(figsize=(10, 4))
plt.plot(raw["timestamp_s"], raw["ax"], label="ax")
plt.plot(raw["timestamp_s"], raw["ay"], label="ay")
plt.plot(raw["timestamp_s"], raw["az"], label="az")
plt.xlabel("Time [s]")
plt.ylabel("Acceleration [m/s²]")
plt.title(f"{DATASET_NAME}: Accelerometer components")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(raw["timestamp_s"], raw["accel_norm"], label="|a|")
plt.axhline(GRAVITY, linestyle="--", label="9.80665 m/s²")
plt.xlabel("Time [s]")
plt.ylabel("Acceleration magnitude [m/s²]")
plt.title(f"{DATASET_NAME}: Acceleration magnitude")
plt.legend()
plt.grid(True)
plt.show()

## 5. Gyroscope check

For static data, `gx`, `gy`, `gz` should be close to zero except bias/noise.

For a 90° controlled rotation, the gyro integral should be around:

```text
90° = 1.57 rad
```

If your integrated angle is around `90` before conversion, your gyro was probably in `deg/s`.

In [ ]:
print("Gyroscope mean [rad/s]:")
print(raw[["gx", "gy", "gz"]].mean())

print("Gyroscope std [rad/s]:")
print(raw[["gx", "gy", "gz"]].std())

plt.figure(figsize=(10, 4))
plt.plot(raw["timestamp_s"], raw["gx"], label="gx")
plt.plot(raw["timestamp_s"], raw["gy"], label="gy")
plt.plot(raw["timestamp_s"], raw["gz"], label="gz")
plt.xlabel("Time [s]")
plt.ylabel("Angular velocity [rad/s]")
plt.title(f"{DATASET_NAME}: Gyroscope components")
plt.legend()
plt.grid(True)
plt.show()

# Gyro-only angle integral: unit/sign sanity check, not a full orientation filter.
angle_x = np.r_[0, np.cumsum(raw["gx"].to_numpy()[1:] * np.diff(raw["timestamp_s"].to_numpy()))]
angle_y = np.r_[0, np.cumsum(raw["gy"].to_numpy()[1:] * np.diff(raw["timestamp_s"].to_numpy()))]
angle_z = np.r_[0, np.cumsum(raw["gz"].to_numpy()[1:] * np.diff(raw["timestamp_s"].to_numpy()))]

plt.figure(figsize=(10, 4))
plt.plot(raw["timestamp_s"], np.degrees(angle_x), label="integral gx")
plt.plot(raw["timestamp_s"], np.degrees(angle_y), label="integral gy")
plt.plot(raw["timestamp_s"], np.degrees(angle_z), label="integral gz")
plt.xlabel("Time [s]")
plt.ylabel("Integrated angle [deg]")
plt.title(f"{DATASET_NAME}: Gyro-only integrated angle sanity check")
plt.legend()
plt.grid(True)
plt.show()

## 6. Coordinate-system worksheet

Fill these in after physical tests.

### Sensor frame definition

| Axis | Physical direction on mounted IMU | Confirmed? |
|---|---|---|
| `+X_sensor` | TODO | TODO |
| `+Y_sensor` | TODO | TODO |
| `+Z_sensor` | TODO | TODO |

### Six-face accelerometer test

| Physical orientation | Expected observation | Actual observation |
|---|---|---|
| board face up | one axis near ±9.8 | TODO |
| board face down | same axis flips sign | TODO |
| right edge up | one axis near ±9.8 | TODO |
| left edge up | same axis flips sign | TODO |
| top edge up | one axis near ±9.8 | TODO |
| bottom edge up | same axis flips sign | TODO |

### Gyroscope sign test

| Motion | Dominant gyro axis | Expected sign by chosen convention | Actual sign |
|---|---|---|---|
| positive rotation about sensor X | `gx` | TODO | TODO |
| positive rotation about sensor Y | `gy` | TODO | TODO |
| positive rotation about sensor Z | `gz` | TODO | TODO |

If the filter output looks wrong, first suspect **unit / axis / sign convention**, then algorithm.

In [ ]:
# Optional: create blank CSV worksheets you can fill in during experiments.
worksheet_dir = DATA_DIR / "worksheets"
worksheet_dir.mkdir(parents=True, exist_ok=True)

sensor_frame = pd.DataFrame({
    "axis": ["+X_sensor", "+Y_sensor", "+Z_sensor"],
    "physical_direction_on_mounted_IMU": ["TODO", "TODO", "TODO"],
    "confirmed": ["TODO", "TODO", "TODO"],
})

accel_six_face = pd.DataFrame({
    "physical_orientation": [
        "board face up", "board face down", "right edge up", "left edge up", "top edge up", "bottom edge up"
    ],
    "expected_observation": [
        "one axis near ±9.8", "same axis flips sign", "one axis near ±9.8", "same axis flips sign", "one axis near ±9.8", "same axis flips sign"
    ],
    "actual_observation": ["TODO"] * 6,
})

gyro_sign = pd.DataFrame({
    "motion": [
        "positive rotation about sensor X", "positive rotation about sensor Y", "positive rotation about sensor Z"
    ],
    "dominant_gyro_axis": ["gx", "gy", "gz"],
    "expected_sign_by_chosen_convention": ["TODO", "TODO", "TODO"],
    "actual_sign": ["TODO", "TODO", "TODO"],
})

sensor_frame.to_csv(worksheet_dir / "sensor_frame_definition.csv", index=False)
accel_six_face.to_csv(worksheet_dir / "six_face_accelerometer_test.csv", index=False)
gyro_sign.to_csv(worksheet_dir / "gyroscope_sign_test.csv", index=False)

print("Created worksheets in", worksheet_dir)
display(sensor_frame)
display(accel_six_face)
display(gyro_sign)

## 7. Load or generate Madgwick output

If `OUT_PATH` exists, this notebook loads it.

If not, it imports `MadgwickFilter` from `madgwick_filter.py`, replays the raw data, and saves the output.

In [ ]:
def run_madgwick_from_raw(raw_df: pd.DataFrame, beta: float = 0.1) -> pd.DataFrame:
    try:
        from madgwick_filter import MadgwickFilter
    except ImportError as exc:
        raise ImportError(
            "Could not import MadgwickFilter. Put madgwick_filter.py in the same folder as this notebook."
        ) from exc

    filt = MadgwickFilter(beta=beta)
    outputs = []
    t = raw_df["timestamp_s"].to_numpy()
    default_dt = np.median(np.diff(t)) if len(t) > 1 else 0.01

    for i, row in raw_df.iterrows():
        dt_i = default_dt if i == 0 else t[i] - t[i - 1]
        if not np.isfinite(dt_i) or dt_i <= 0:
            dt_i = default_dt

        result = filt.update(
            accel_x=float(row["ax"]),
            accel_y=float(row["ay"]),
            accel_z=float(row["az"]),
            gyro_x=float(row["gx"]),
            gyro_y=float(row["gy"]),
            gyro_z=float(row["gz"]),
            dt=float(dt_i),
        )
        result["timestamp_s"] = float(row["timestamp_s"])
        outputs.append(result)

    out_df = pd.DataFrame(outputs)

    # Add standardized angle columns.
    for rad_col, deg_col in [("roll", "roll_deg"), ("pitch", "pitch_deg"), ("yaw", "yaw_deg")]:
        if rad_col in out_df.columns:
            out_df[f"{rad_col}_rad"] = out_df[rad_col]
            out_df[deg_col] = np.degrees(out_df[rad_col])

    # Rename camelCase outputs from the Python Madgwick class to snake_case.
    out_df = out_df.rename(columns={
        "gravityX": "gravity_x",
        "gravityY": "gravity_y",
        "gravityZ": "gravity_z",
        "userAccelX": "user_accel_x",
        "userAccelY": "user_accel_y",
        "userAccelZ": "user_accel_z",
    })

    preferred = [
        "timestamp_s", "qw", "qx", "qy", "qz",
        "roll_rad", "pitch_rad", "yaw_rad",
        "roll_deg", "pitch_deg", "yaw_deg",
        "gravity_x", "gravity_y", "gravity_z",
        "user_accel_x", "user_accel_y", "user_accel_z",
    ]
    existing = [c for c in preferred if c in out_df.columns]
    others = [c for c in out_df.columns if c not in existing]
    return out_df[existing + others]

if OUT_PATH.exists():
    out = pd.read_csv(OUT_PATH)
    out.columns = [c.strip() for c in out.columns]
    print("Loaded existing Madgwick output:", OUT_PATH)
else:
    out = run_madgwick_from_raw(raw, beta=BETA)
    OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(OUT_PATH, index=False)
    print("Generated and saved Madgwick output:", OUT_PATH)

out.head()

## 8. Euler angle output

Expected behaviours:

| Dataset | Expected output |
|---|---|
| static | roll/pitch stable; yaw may drift slowly |
| roll test | roll changes most clearly |
| pitch test | pitch changes most clearly |
| yaw test | yaw changes but may drift |
| epley-like | clear step-like orientation transitions |

In [ ]:
# Handle alternative names if needed.
if "roll_deg" not in out.columns and "roll" in out.columns:
    out["roll_deg"] = np.degrees(out["roll"])
if "pitch_deg" not in out.columns and "pitch" in out.columns:
    out["pitch_deg"] = np.degrees(out["pitch"])
if "yaw_deg" not in out.columns and "yaw" in out.columns:
    out["yaw_deg"] = np.degrees(out["yaw"])

plt.figure(figsize=(11, 5))
plt.plot(out["timestamp_s"], out["roll_deg"], label="roll")
plt.plot(out["timestamp_s"], out["pitch_deg"], label="pitch")
plt.plot(out["timestamp_s"], out["yaw_deg"], label="yaw")
plt.xlabel("Time [s]")
plt.ylabel("Angle [deg]")
plt.title(f"{DATASET_NAME}: Madgwick Euler angles")
plt.legend()
plt.grid(True)
plt.show()

out[["timestamp_s", "roll_deg", "pitch_deg", "yaw_deg"]].describe()

## 9. Quaternion norm check

A valid orientation quaternion should satisfy:

```text
sqrt(qw^2 + qx^2 + qy^2 + qz^2) ≈ 1
```

In [ ]:
q_cols = ["qw", "qx", "qy", "qz"]
missing_q = [c for c in q_cols if c not in out.columns]
if missing_q:
    raise ValueError(f"Missing quaternion columns: {missing_q}")

out["q_norm"] = np.sqrt(out["qw"]**2 + out["qx"]**2 + out["qy"]**2 + out["qz"]**2)

print("Quaternion norm statistics:")
print(out["q_norm"].describe())
print("Max deviation from 1:", np.max(np.abs(out["q_norm"] - 1)))

plt.figure(figsize=(10, 4))
plt.plot(out["timestamp_s"], out["q_norm"])
plt.axhline(1.0, linestyle="--", label="norm = 1")
plt.xlabel("Time [s]")
plt.ylabel("Quaternion norm")
plt.title(f"{DATASET_NAME}: Quaternion norm")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(out["timestamp_s"], out["qw"], label="qw")
plt.plot(out["timestamp_s"], out["qx"], label="qx")
plt.plot(out["timestamp_s"], out["qy"], label="qy")
plt.plot(out["timestamp_s"], out["qz"], label="qz")
plt.xlabel("Time [s]")
plt.ylabel("Quaternion component")
plt.title(f"{DATASET_NAME}: Quaternion components")
plt.legend()
plt.grid(True)
plt.show()

## 10. Estimated gravity vs measured acceleration

During static periods, measured acceleration should be close to estimated gravity.

During dynamic movement, they will differ because the accelerometer measures gravity plus linear acceleration.

In [ ]:
gravity_cols = ["gravity_x", "gravity_y", "gravity_z"]
if not all(c in out.columns for c in gravity_cols):
    print("Gravity columns not found; skipping this section.")
else:
    for axis, measured, gravity_col in [
        ("X", "ax", "gravity_x"),
        ("Y", "ay", "gravity_y"),
        ("Z", "az", "gravity_z"),
    ]:
        plt.figure(figsize=(10, 4))
        plt.plot(raw["timestamp_s"], raw[measured], label=f"measured a{axis.lower()}")
        plt.plot(out["timestamp_s"], out[gravity_col], label=f"estimated gravity_{axis.lower()}")
        plt.xlabel("Time [s]")
        plt.ylabel("Acceleration [m/s²]")
        plt.title(f"{DATASET_NAME}: measured acceleration vs estimated gravity ({axis})")
        plt.legend()
        plt.grid(True)
        plt.show()

## 11. User acceleration check

```text
user_accel = measured_accel - estimated_gravity
```

For static data, user acceleration should be close to zero. For manoeuvres, spikes are expected during transitions.

In [ ]:
user_cols = ["user_accel_x", "user_accel_y", "user_accel_z"]
if not all(c in out.columns for c in user_cols):
    print("User acceleration columns not found; skipping this section.")
else:
    out["user_accel_norm"] = np.sqrt(
        out["user_accel_x"]**2 + out["user_accel_y"]**2 + out["user_accel_z"]**2
    )
    print(out[user_cols + ["user_accel_norm"]].describe())

    plt.figure(figsize=(10, 4))
    plt.plot(out["timestamp_s"], out["user_accel_x"], label="user_accel_x")
    plt.plot(out["timestamp_s"], out["user_accel_y"], label="user_accel_y")
    plt.plot(out["timestamp_s"], out["user_accel_z"], label="user_accel_z")
    plt.xlabel("Time [s]")
    plt.ylabel("User acceleration [m/s²]")
    plt.title(f"{DATASET_NAME}: User acceleration")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(out["timestamp_s"], out["user_accel_norm"], label="|user acceleration|")
    plt.xlabel("Time [s]")
    plt.ylabel("User acceleration magnitude [m/s²]")
    plt.title(f"{DATASET_NAME}: User acceleration magnitude")
    plt.legend()
    plt.grid(True)
    plt.show()

## 12. Static-test summary

Use this mainly for `static_30s`.

Suggested checks:

| Check | Expected |
|---|---|
| mean acceleration norm | close to 9.81 m/s² |
| gyro mean | close to 0 rad/s |
| quaternion norm | close to 1 |
| roll/pitch | stable |
| yaw | may drift slowly |
| user acceleration | close to 0 during static periods |

In [ ]:
summary = {
    "dataset": DATASET_NAME,
    "duration_s": float(raw["timestamp_s"].iloc[-1] - raw["timestamp_s"].iloc[0]),
    "mean_sampling_rate_hz": float(1 / np.mean(np.diff(raw["timestamp_s"]))) if len(raw) > 1 else np.nan,
    "accel_norm_mean_mps2": float(raw["accel_norm"].mean()),
    "accel_norm_std_mps2": float(raw["accel_norm"].std()),
    "gyro_x_mean_rads": float(raw["gx"].mean()),
    "gyro_y_mean_rads": float(raw["gy"].mean()),
    "gyro_z_mean_rads": float(raw["gz"].mean()),
    "q_norm_max_deviation": float(np.max(np.abs(out["q_norm"] - 1))) if "q_norm" in out.columns else np.nan,
    "roll_range_deg": float(out["roll_deg"].max() - out["roll_deg"].min()) if "roll_deg" in out.columns else np.nan,
    "pitch_range_deg": float(out["pitch_deg"].max() - out["pitch_deg"].min()) if "pitch_deg" in out.columns else np.nan,
    "yaw_range_deg": float(out["yaw_deg"].max() - out["yaw_deg"].min()) if "yaw_deg" in out.columns else np.nan,
}
summary_df = pd.DataFrame([summary])
display(summary_df)

summary_path = DATA_DIR / f"{DATASET_NAME}_validation_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved summary to", summary_path)

## 13. Manoeuvre-level interpretation

Use this for `epley_like`, `dix_hallpike_like`, or other controlled motion recordings.

BPPV-related orientation questions the IMU can help answer:

- Did the head rotate approximately 45° toward the tested side?
- Did the head reach extension/supine posture?
- Did the head rotate approximately 90° between manoeuvre positions?
- How long was each position held?
- Was the movement sequence correct?

Important limitation: this IMU pipeline estimates **head orientation**. It does not directly observe nystagmus or diagnose BPPV.

In [ ]:
# Optional event markers. Fill this in manually after inspecting your recording.
# Example:
# EVENT_TIMES = {"start": 0.0, "pos1": 5.2, "pos2": 35.0, "pos3": 65.0}
EVENT_TIMES = {}

plt.figure(figsize=(11, 5))
plt.plot(out["timestamp_s"], out["roll_deg"], label="roll")
plt.plot(out["timestamp_s"], out["pitch_deg"], label="pitch")
plt.plot(out["timestamp_s"], out["yaw_deg"], label="yaw")
for label, time_s in EVENT_TIMES.items():
    plt.axvline(time_s, linestyle="--")
    plt.text(time_s, plt.ylim()[1], label, rotation=90, va="top")
plt.xlabel("Time [s]")
plt.ylabel("Angle [deg]")
plt.title(f"{DATASET_NAME}: Manoeuvre orientation with optional event markers")
plt.legend()
plt.grid(True)
plt.show()

## 14. Supervisor update wording

> The XIAO Seeed IMU can stream accelerometer and gyroscope data in real time. The supervisor-provided Madgwick filter is used as a baseline orientation estimator. I standardized timestamps to seconds, acceleration to m/s², and gyroscope measurements to rad/s before filtering. I am validating the pipeline using static, single-axis rotation, and manoeuvre-like recordings. The current focus is to confirm unit scaling, sampling stability, sensor-axis signs, quaternion normalization, and roll/pitch/yaw behaviour before implementing a Kalman/EKF comparison.

Next steps:

1. Collect `static_30s`, `roll_test`, `pitch_test`, `yaw_test`, and `epley_like` datasets.
2. Fill in the coordinate-system worksheets.
3. Decide the final sensor-to-head mounting convention.
4. Use this Madgwick pipeline as baseline/reference for Kalman/EKF development.